In [1]:
# ==============================================================================
# BLOCK 1: INSTALL DEPENDENCIES AND IMPORT LIBRARIES
# ==============================================================================
!pip install -q python-chess

import io
import chess
import chess.pgn
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Set device and random seed for reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

print(f"[INFO] PyTorch Version: {torch.__version__}")
print(f"[INFO] Target Device: {device}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[INFO] PyTorch Version: 2.14.1+cu130
[INFO] Target Device: cuda


In [2]:
# ==============================================================================
# BLOCK 2: BOARD TO TENSOR ENCODING (14 x 8 x 8)
# ==============================================================================
def board_to_tensor(board: chess.Board) -> np.ndarray:
    """
    Converts a chess.Board object into a (14, 8, 8) float32 numpy array.
    Channels 0-5: White pieces (P, N, B, R, Q, K)
    Channels 6-11: Black pieces (p, n, b, r, q, k)
    Channel 12: Turn indicator (1.0 for White, 0.0 for Black)
    Channel 13: Valid destination squares for legal moves
    """
    tensor = np.zeros((14, 8, 8), dtype=np.float32)

    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row = 7 - (square // 8)
        col = square % 8

        # Piece type runs 1 to 6 (Pawn to King)
        channel = piece.piece_type - 1
        if piece.color == chess.BLACK:
            channel += 6

        tensor[channel, row, col] = 1.0

    # Channel 12: Player turn
    if board.turn == chess.WHITE:
        tensor[12, :, :] = 1.0

    # Channel 13: Legal destination squares
    for move in board.legal_moves:
        to_sq = move.to_square
        row = 7 - (to_sq // 8)
        col = to_sq % 8
        tensor[13, row, col] = 1.0

    return tensor


# Test the encoding on the starting board
sample_board = chess.Board()
sample_tensor = board_to_tensor(sample_board)

print("[INFO] Board encoding complete.")
print(f"[DEBUG] Output Tensor Shape: {sample_tensor.shape}")
print(f"[DEBUG] Total active pieces on starting board: {int(sample_tensor[:12].sum())}")
print(f"[DEBUG] Current Turn Channel Value: {sample_tensor[12, 0, 0]}")

[INFO] Board encoding complete.
[DEBUG] Output Tensor Shape: (14, 8, 8)
[DEBUG] Total active pieces on starting board: 32
[DEBUG] Current Turn Channel Value: 1.0


In [3]:
# ==============================================================================
# BLOCK 3: MOVE ENCODING AND DECODING (4096 ACTION SPACE)
# ==============================================================================
def encode_move(move: chess.Move) -> int:
    """
    Encodes a chess.Move into a flat integer index [0, 4095].
    Formula: from_square * 64 + to_square
    """
    return move.from_square * 64 + move.to_square


def decode_move(index: int, board: chess.Board) -> chess.Move:
    """
    Decodes an integer index [0, 4095] back into a legal chess.Move.
    Automatically handles standard Queen promotion if applicable.
    """
    from_square = index // 64
    to_square = index % 64

    # Check if a pawn is moving to the last rank for promotion
    piece = board.piece_at(from_square)
    promotion = None
    if piece and piece.piece_type == chess.PAWN:
        if (piece.color == chess.WHITE and chess.square_rank(to_square) == 7) or \
           (piece.color == chess.BLACK and chess.square_rank(to_square) == 0):
            promotion = chess.QUEEN

    return chess.Move(from_square, to_square, promotion=promotion)


# Test move encoding and decoding
test_move = chess.Move.from_uci("e2e4")
encoded_idx = encode_move(test_move)
decoded_move = decode_move(encoded_idx, sample_board)

print(f"[INFO] Testing Move Tokenizer:")
print(f"[DEBUG] Original Move: {test_move.uci()} -> Encoded Index: {encoded_idx}")
print(f"[DEBUG] Decoded Move: {decoded_move.uci()} (Match: {test_move == decoded_move})")

[INFO] Testing Move Tokenizer:
[DEBUG] Original Move: e2e4 -> Encoded Index: 796
[DEBUG] Decoded Move: e2e4 (Match: True)


In [4]:
# ==============================================================================
# BLOCK 4: PGN DATA PARSER
# ==============================================================================
SAMPLE_PGN = """
[Event "F/S Return Match"]
[Site "Belgrade, Serbia JUG"]
[Date "1992.11.04"]
[Round "29"]
[White "Fischer, Robert J."]
[Black "Spassky, Boris V."]
[Result "1/2-1/2"]

1. e4 e5 2. Nf3 Nc6 3. Bb5 a6 4. Ba4 Nf6 5. O-O Be7 6. Re1 b5 7. Bb3 d6 8. c3 O-O
9. h3 Nb8 10. d4 Nbd7 11. c4 c6 12. cxb5 axb5 13. Nc3 Bb7 14. Bg5 b4 15. Nb1 h6
16. Bh4 c5 17. dxe5 Nxe4 18. Bxe7 Qxe7 19. exd6 Qf6 20. Nbd2 Nxd6 1/2-1/2
"""

def parse_pgn_string(pgn_text: str):
    """
    Reads PGN string and extracts (board_tensor, move_index) pairs.
    """
    states = []
    labels = []
    pgn_io = io.StringIO(pgn_text)
    game_count = 0

    while True:
        game = chess.pgn.read_game(pgn_io)
        if game is None:
            break

        game_count += 1
        board = game.board()

        for move in game.mainline_moves():
            states.append(board_to_tensor(board))
            labels.append(encode_move(move))
            board.push(move)

    X = np.array(states, dtype=np.float32)
    y = np.array(labels, dtype=np.int64)

    print(f"[INFO] PGN Parsing Finished. Games processed: {game_count}")
    print(f"[INFO] Total positions extracted: {len(X)}")
    return X, y


X_data, y_data = parse_pgn_string(SAMPLE_PGN)

[INFO] PGN Parsing Finished. Games processed: 1
[INFO] Total positions extracted: 40


In [5]:
# ==============================================================================
# BLOCK 5: PYTORCH DATASET AND DATALOADER
# ==============================================================================
class BozonChessDataset(Dataset):
    """
    Custom PyTorch Dataset for BOZON Chess Engine.
    """
    def __init__(self, states: np.ndarray, labels: np.ndarray):
        self.states = torch.tensor(states, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.states)

    def __getitem__(self, idx: int):
        return self.states[idx], self.labels[idx]


# Initialize Dataset and DataLoader
dataset = BozonChessDataset(X_data, y_data)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Fetch one sample batch to verify pipeline integrity
batch_states, batch_labels = next(iter(dataloader))

print("[INFO] PyTorch DataLoader initialized successfully.")
print(f"[DEBUG] Batch States Shape: {batch_states.shape} | Dtype: {batch_states.dtype}")
print(f"[DEBUG] Batch Labels Shape: {batch_labels.shape} | Dtype: {batch_labels.dtype}")
print(f"[DEBUG] Sample Label Indices in Batch: {batch_labels.tolist()}")
print("[SUCCESS] BOZON v0.1 Data Pipeline is ready for training.")

[INFO] PyTorch DataLoader initialized successfully.
[DEBUG] Batch States Shape: torch.Size([8, 14, 8, 8]) | Dtype: torch.float32
[DEBUG] Batch Labels Shape: torch.Size([8]) | Dtype: torch.int64
[DEBUG] Sample Label Indices in Batch: [262, 658, 3902, 3956, 324, 3828, 353, 75]
[SUCCESS] BOZON v0.1 Data Pipeline is ready for training.
